In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
llm = ChatOllama(model="gemma4:e2b", base_url="http://127.0.0.1:11434")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 저는 잘 지내고 있습니다. 😊\n\n**당신은 잘 지내셨나요?**', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:37:06.4045649Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25558750200, 'load_duration': 24886926100, 'prompt_eval_count': 21, 'prompt_eval_duration': 288259000, 'eval_count': 23, 'eval_duration': 380430000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80b-a28b-75a1-9c17-2e46ae2eb0eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 23, 'total_tokens': 44})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:37:22.0721841Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4067555800, 'load_duration': 10711100, 'prompt_eval_count': 157, 'prompt_eval_duration': 940708000, 'eval_count': 297, 'eval_duration': 3109051000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80c-33b3-7f40-8fb2-41897faef8f7-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': 'cfb7f391-cfcd-40cc-a07a-079c59666587', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 297, 'total_tokens': 454})]


In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': 'Busan', 'timezone': 'Asia/Seoul'}
Asia/Seoul (Busan) 현재시각 2026-09-22 16:37:22 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:37:22.0721841Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4067555800, 'load_duration': 10711100, 'prompt_eval_count': 157, 'prompt_eval_duration': 940708000, 'eval_count': 297, 'eval_duration': 3109051000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80c-33b3-7f40-8fb2-41897faef8f7-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': 'cfb7f391-cfcd-40cc-a07a-079c59666587', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 297, 'total_tokens': 454}),
 ToolMessage(content='Asia/Seoul (Busan) 현재시각 2026-09-22 16:37:22 ', nam

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 **2026년 9월 22일 16시 37분 22초**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:37:23.5748474Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1355702500, 'load_duration': 4745000, 'prompt_eval_count': 227, 'prompt_eval_duration': 83188000, 'eval_count': 133, 'eval_duration': 1258066000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80c-442a-74e1-8d53-e15b1fd7cea4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 133, 'total_tokens': 360})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")


In [9]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수
    Args:
        stock_history_input.ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
        stock_history_input.period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")
    """
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [10]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={} response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:37:57.3617414Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6828567300, 'load_duration': 6454500, 'prompt_eval_count': 389, 'prompt_eval_duration': 842340000, 'eval_count': 607, 'eval_duration': 5964912000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'} id='lc_run--01a0c80c-b2c3-7292-b3f7-53b9129f9554-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}, 'id': '1addb7ca-33a6-4c21-b26f-8b791fb75295', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 389, 'output_tokens': 607, 'total_tokens': 996}


In [11]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-08-24 00:00:00-04:00 | 361.41 | 363.24 | 348.26 |  348.95 | 3.91905e+07 |           0 |              0 |\n| 2026-08-25 00:00:00-04:00 | 349.58 | 357    | 349.2  |  350.25 | 2.98167e+07 |           0 |              0 |\n| 2026-08-26 00:00:00-04:00 | 345.27 | 351.93 | 342.53 |  345.82 | 2.8573e+07  |           0 |              0 |\n| 2026-08-27 00:00:00-04:00 | 346.16 | 355.74 | 345.45 |  354.81 | 3.01362e+07 |           0 |              0 |\n| 2026-08-28 00:00:00-04:00 | 357.1  | 358.8  | 345.2  |  348.75 | 3.29722e+07 |           0 |              0 |\n| 2026-08-31 00:00:00-04:00 | 347.21 | 368.92 | 347.15 |  367.95 | 6.17323e+07 |           0 |              0 |\n| 2026-09-01 00:00:00-04:0

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전(8월 24일경) 대비 테슬라(TSLA) 주가는 **올랐습니다**.\n\n약 한 달 전의 종가는 대략 348.95달러였던 반면, 가장 최근 데이터(9월 21일)의 종가는 375.30달러로 상승했습니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:38:06.0895172Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3618365200, 'load_duration': 6362400, 'prompt_eval_count': 2074, 'prompt_eval_duration': 692501000, 'eval_count': 307, 'eval_duration': 2897373000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80c-e165-75b0-9964-f8987250ac3a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2074, 'output_tokens': 307, 'total_tokens': 2381})

## 08-4 스트림 출력

In [13]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||네|,| 저는| 잘| 지|내|고| 있습니다|.| 질문|해| 주|셔서| 감사합니다|.|

|한국| 사회|는| 눈|부|신| 경제| 성장|과| 놀|라운| 발|전을| 이루|었|지만|,| 동시에| 복|잡|하고| 다양한| 사회|적|,| 구조|적| 문제|점|들을| 안|고| 있습니다|.| 이러한| 문제|들은| 개인|적인| 차|원을| 넘어| 사회| 시스템| 전체|에| 걸|쳐| 깊|이| 뿌|리|내|리고| 있으며|,| 세|대|별|,| 계|층|별|로| 다|르게| 경험|되고| 있습니다|.|

|주|요|하게| 논|의|되는| 한국| 사회|의| 문제|점|들을| 몇| 가지| 영역|으로| 나누|어| 말씀|드리|겠습니다|.|

|---|

|###| |1|.| 극|심|한| 경쟁| 사회|와| 삶|의| 질| 저|하|

|한국| 사회|의| 가장| 두|드|러|진| 특징| 중| 하나|는| **|극|심|한| 경쟁| 사회|**|입니다|.| 학|력|,| 직|장|,| 부동산| 등| 모든| 영역|에서| 성공|을| 위해| 끊|임|없이| 경쟁|해야| 한다는| 압|박|감이| 존재|합니다|.|

|*| 

In [14]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk []
chunk type:  <class 'langchain_core.messages.ai.AI

In [15]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:38:47.2784995Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3327469200, 'load_duration': 5496300, 'prompt_eval_count': 296, 'prompt_eval_duration': 371279000, 'eval_count': 323, 'eval_duration': 2920235000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80d-836d-7b03-bfcd-10300f592f0a', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': '48a03e6d-1474-4d74-bba3-491863c697ef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 296, 'output_tokens': 323, 'total_tokens': 619}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"location": "Busan", "timezone": "Asia/Seoul"}', 'id': '48a03e6d-1474-4d74-bba3-491863c697ef', 'index': None, 'type': 'tool_call_chunk'}], chunk_position='last')

In [16]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': 'Busan', 'timezone': 'Asia/Seoul'}
Asia/Seoul (Busan) 현재시각 2026-09-22 16:39:09 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-22T07:38:47.2784995Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3327469200, 'load_duration': 5496300, 'prompt_eval_count': 296, 'prompt_eval_duration': 371279000, 'eval_count': 323, 'eval_duration': 2920235000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0c80d-836d-7b03-bfcd-10300f592f0a', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': '48a03e6d-1474-4d74-bba3-491863c697ef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 296, 'output_tokens': 323, 'total_tokens': 619}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"location": "B

In [17]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||부산|은| 현재| **|2|0|2|6|년| |9|월| |2|2|일| |1|6|시| |3|9|분| |0|9|초|**|입니다|.|||